# Генерація тестового набору даних

Для оцінки якості пошуку та роботи RAG знадобиться тестовий набір даних, що містить запити. Для кожного запиту в цьому наборі ми знаємо, який документ у базі знань є релевантним до нього. Під час пошуку ми перевіряємо, чи містять результати  правильний документ.

Існує кілька способів отримання тестових даних:

- Люди-анотатори переглядають документи та пишуть запити (найкраща якість але висока ціна);
- Збирати запити реальних користувачів та маркувати їх (потрібна запущена система);
- Генерація синтетичних даних за допомогою LLM (що ми і зробимо).

Ми не маємо ще робочої системи, тому використовуватимемо LLM для генерації питань для тестів. Для кожного документа FAQ ми просимо LLM створити 5 питань, на які цей документ відповість. Тоді ми будемо знати, що для кожного згенерованого питання вихідний документ є правильною відповіддю.

Але треба мати на увазі, що згенеровані питання можуть бути занадто схожими на оригінальний текст FAQу.  Це завищує показники. Тому якомога швидше треба починати збирати запити від реальних користувачів і використовувати їх для перевірки створеної системи.

## Завантаження даних FAQу.

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

Ми генеруватимемо питання лише для поширених запитань LLM Zoomcamp. Повний набір поширених запитань містить документи з кількох курсів. Генерація п'яти питань для кожного документа займе більше часу та коштуватиме дорожче.

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

Далі ми використовуємо тільки ці документи. Отже перейменуємо їх на documents.

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Кожен документ має поле id. Ідентифікатор стає міткою в нашому наборі даних. Під час оцінювання пошуку перевіряється, чи повертає пошук документ із цим ідентифікатором.

Ось чому кожен запис потребує стабільного ідентифікатора. Якщо ви не можете однозначно ідентифікувати документ, ви не можете сказати, чи пошук знайшов саме той, що треба. Під час створення власного набору для оцінювання, спочатку призначаємо ідентифікатор кожному запису у нашій базі знань.

## Генерація питань із структурованим виводом

Ми використовуємо LLM, щоб сгенерувати питання для кожного документа. І ми застосовуємо структурований вивід. Тобто просимо LLM повернути дані у певному форматі, а не у вільному тексті. Наприклад, замість того, щоб отримувати абзац, який містить питання, ми можемо запитувати об'єкт Python з полем questions.

Це корисно, коли код оброблятиме вивід. Модель щоразу повертає ту саму структуру. Ми можемо отримати доступ до згенерованих питань безпосередньо, замість того, щоб вручну аналізувати текст.

Нам потрібен вивід у вигляді списку рядків, тому ми визначаємо цю структуру за допомогою моделі Pydantic:

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

Інструкіця для LLM.

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

Ми просимо LLM використовувати формулювання, відмінне від оригінального документа. Це робить оцінку більш реалістичною — реальні користувачі не формулюватимуть свої запитання так само, як у FAQ.

In [6]:
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

Підготуємо документ у форматі JSON.

In [7]:
import json

user_prompt = json.dumps(doc)

In [10]:
interaction = client.interactions.create(
        model="gemini-3.1-flash-lite",
        system_instruction=data_gen_instructions,
        input=user_prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": Questions.model_json_schema()
        }
    )

result = Questions.model_validate_json(interaction.output_text)
print(result)

questions=['Is it too late to jump into the course if I just found out about it today?', 'If I start the material late, is there any chance I can still earn the completion certificate?', 'Are there any penalties or restrictions for joining the cohort after it has already kicked off?', 'What do I need to do regarding the final project if I’m joining the program this late?', "Can I still get certified if I'm catching up on the lessons now, or have I missed the deadline?"]
